In [ ]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
:set -XLambdaCase
import Data.List (sortBy, nub, intercalate)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe, mapMaybe)
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Uncertainty & Randomness"

# ❓ Uncertainty & Randomness in Haskell

> Категорное представление случайности и неопределённости: монады Гири, модальные логики, нечёткая математика, стохастическое программирование, битопологические пространства

| № | Тема |
|---|------|
| 1 | Неопределённость как функтор |
| 2 | Монада дискретных распределений |
| 3 | Монада Гири и вероятностные меры |
| 4 | Нечёткая математика и нечёткие множества |
| 5 | Модальные логики и фреймы Крипке |
| 6 | Стохастическое программирование: DSL для байесовских сетей |
| 7 | Марковские цепи в категории Клейсли |
| 8 | Категорная картина: стохастические матрицы и теория меры |
| 9 | Теория Пытьева → [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) |


# 1. Неопределённость как Функтор

| Тип | Семантика |
|------|----------|
| `Maybe a` | отсутствие/наличие |
| `Either e a` | ошибка/успех |
| `[a]` | множественность |
| `Validation` | накопл. ошибок |

### 🔸 эндофункторы в **Hask**

In [ ]:
-- S1: Uncertainty as Functor

data Validation e a = Failure e | Success a deriving (Show)

instance Functor (Validation e) where
  fmap _ (Failure e) = Failure e
  fmap f (Success a) = Success (f a)

instance Semigroup e => Applicative (Validation e) where
  pure = Success
  Failure e1 <*> Failure e2 = Failure (e1 <> e2)
  Failure e  <*> _          = Failure e
  _          <*> Failure e  = Failure e
  Success f  <*> Success x  = Success (f x)

safeDiv :: Int -> Int -> Maybe Int
safeDiv _ 0 = Nothing
safeDiv a b = Just (a `div` b)

parseAge :: String -> Either String Int
parseAge s = case reads s of
  [(n, "")] | n >= 0 && n <= 150 -> Right n
  [(_, "")]  -> Left ("out of range: " ++ s)
  _           -> Left ("not a number: " ++ s)

demo_sec1 :: IO ()
demo_sec1 = do
  putStrLn "\n=== S1: Uncertainty as Functor ==="
  putStrLn "\n-- Maybe:"
  print (safeDiv 10 2)
  print (safeDiv 10 0)
  putStrLn "\n-- Either:"
  print (parseAge "25")
  print (parseAge "abc")
  putStrLn "\n-- Validation:"
  let v1 = Success (+) <*> Success 1 <*> Success 2 :: Validation [String] Int
      v2 = Success (+) <*> Failure ["e1"] <*> Failure ["e2"] :: Validation [String] Int
  print v1
  print v2
  putStrLn "\n-- List nondeterminism:"
  let ms = [ (x, y) | x <- [1,2,3::Int], y <- [1,2,3] ]
  putStrLn ("3x3 = " ++ show (length ms))
  print (take 4 ms)

demo_sec1

# 2. Монада Дискретных Распределений

`Dist a` = кон. вероят. распределение.

### 🔸 Kleisli(Dist) = **FinStoch**

In [ ]:
-- S2: Dist Monad

newtype Dist a = Dist { getProbs :: [(a, Double)] } deriving (Show)

instance Functor Dist where
  fmap f (Dist ps) = Dist [ (f a, p) | (a, p) <- ps ]

instance Applicative Dist where
  pure x = Dist [(x, 1.0)]
  Dist fs <*> Dist xs = Dist [ (f x, p*q) | (f, p) <- fs, (x, q) <- xs ]

instance Monad Dist where
  return = pure
  Dist ps >>= f = Dist [ (b, p*q) | (a, p) <- ps, (b, q) <- getProbs (f a) ]

uniform :: [a] -> Dist a
uniform xs = Dist [ (x, 1.0 / fromIntegral (length xs)) | x <- xs ]

certainly :: a -> Dist a
certainly x = Dist [(x, 1.0)]

condition :: (a -> Bool) -> Dist a -> Dist a
condition p (Dist ps) =
  let fl = [ (a, w) | (a, w) <- ps, p a ]
      t  = sum (map snd fl)
  in  Dist [ (a, w/t) | (a, w) <- fl ]

expectation :: Dist Double -> Double
expectation (Dist ps) = sum [ v * p | (v, p) <- ps ]

showDist :: Show a => Dist a -> String
showDist (Dist ps) =
  unlines [ "  " ++ show v ++ " -> " ++ show (round (p*100) :: Int) ++ "%"
           | (v, p) <- sortBy (comparing (Down . snd)) ps ]

data CoinType = Fair | Biased deriving (Show, Eq, Ord)

priorCoin :: Dist CoinType
priorCoin = Dist [(Fair, 0.5), (Biased, 0.5)]

flipCoin :: CoinType -> Dist String
flipCoin Fair   = Dist [("H", 0.5), ("T", 0.5)]
flipCoin Biased = Dist [("H", 0.9), ("T", 0.1)]

posteriorGivenH :: Dist CoinType
posteriorGivenH = condition (const True) $ do
  ct <- priorCoin
  oc <- flipCoin ct
  if oc == "H" then return ct else Dist []

demo_sec2 :: IO ()
demo_sec2 = do
  putStrLn "\n=== S2: Dist Monad ==="
  putStrLn "\n-- Uniform die:"
  putStr (showDist (uniform [1..6 :: Int]))
  let twoD = do { a <- uniform [1..6::Int]; b <- uniform [1..6::Int]; return (a+b) }
  putStrLn "\n-- Two dice:"
  putStr (showDist twoD)
  putStrLn "\n-- Bayesian coin:"
  putStr (showDist posteriorGivenH)
  let wt = Dist [("sunny", 0.6), ("rainy", 0.4)]
      md w = if w == "sunny"
               then Dist [("happy", 0.8), ("sad", 0.2)]
               else Dist [("happy", 0.3), ("sad", 0.7)]
  putStrLn "\n-- Weather->mood:"
  putStr (showDist (wt >>= md))

demo_sec2

# 3. Монада Гири

Markov-ядро = морфизм `X -> G(Y)` в **Kleisli(G)**.

### 🔸 `>>=` = интеграция по ядру

In [ ]:
-- S3: Giry Monad

type Kernel a b = a -> Dist b

composeK :: Kernel a b -> Kernel b c -> Kernel a c
composeK k1 k2 a = k1 a >>= k2

idKernel :: Kernel a a
idKernel = return

gaussianApprox :: Double -> Double -> Int -> Dist Double
gaussianApprox mu sigma n =
  let st  = 6 * sigma / fromIntegral n
      lo  = mu - 3 * sigma
      pts = [ lo + st * fromIntegral i | i <- [0..n-1] ]
      pdf x = exp (-(x-mu)^(2::Int) / (2*sigma^(2::Int))) / (sigma * sqrt (2*pi))
      ws  = map pdf pts
      tot = sum ws
  in  Dist (zip pts (map (/tot) ws))

betaApprox :: Double -> Double -> Int -> Dist Double
betaApprox a b n =
  let pts = [ fromIntegral i / fromIntegral n | i <- [1..n-1] ]
      pdf x = x**(a-1) * (1-x)**(b-1)
      ws  = map pdf pts
      tot = sum ws
  in  Dist (zip pts (map (/tot) ws))

rwKernel :: Double -> Kernel Double Double
rwKernel sz x = uniform [x - sz, x, x + sz]

iterateK :: Int -> Kernel a a -> Kernel a a
iterateK 0 _ = idKernel
iterateK n k = k `composeK` iterateK (n-1) k

demo_sec3 :: IO ()
demo_sec3 = do
  putStrLn "\n=== S3: Giry Monad ==="
  let g    = gaussianApprox 0 1 20
      top5 = take 5 . sortBy (comparing (Down . snd)) $ getProbs g
  putStrLn "\n-- Gaussian N(0,1):"
  mapM_ (\(x,p) -> putStrLn ("  x~" ++ show (round (x*10) :: Int)
    ++ "/10 p=" ++ show (round (p*100) :: Int) ++ "%")) top5
  let bD = betaApprox 2 5 100
  putStrLn ("\n-- Beta(2,5) mean ~ " ++ show (round (expectation bD * 1000) :: Int)
    ++ "/1000 (true: 286/1000)")
  putStrLn "\n-- RW 3 steps from 0:"
  let d3 = iterateK 3 (rwKernel 1.0) 0.0
  mapM_ (\(x,p) -> putStrLn ("  x=" ++ show x ++ " p=" ++ show (round (p*100) :: Int) ++ "%"))
        (sortBy (comparing fst) (getProbs d3))
  putStrLn "\n-- Kleisli die->coin:"
  let kD = uniform [1..6::Int]
      kC n = if even n then Dist [("H", 0.8), ("T", 0.2)]
                        else Dist [("H", 0.3), ("T", 0.7)]
  putStr (showDist (kD >>= kC))

demo_sec3

# 4. Нечёткая Математика

Fuzzy set = `X -> [0,1]`. Решётка `([0,1], min, max)`.

| t-норма min | `min(a,b)` | AND |
|---|---|---|
| t-норма prod | `a*b` | AND (prob) |
| t-конорма max | `max(a,b)` | OR |
| Lukasiewicz | `min(1,1-a+b)` | импл. |

### 🔸 `[0,1]` = комплектная замкн. подкат. Hask

In [ ]:
-- S4: Fuzzy Mathematics

newtype FuzzySet a = FuzzySet { fmu :: a -> Double }

tMin :: Double -> Double -> Double
tMin a b = min a b

tProd :: Double -> Double -> Double
tProd a b = a * b

tLuka :: Double -> Double -> Double
tLuka a b = max 0 (a + b - 1)

sMax :: Double -> Double -> Double
sMax a b = max a b

fuzzyNot :: Double -> Double
fuzzyNot x = 1 - x

tall :: Int -> Double
tall h
  | h < 160   = 0.0
  | h > 190   = 1.0
  | otherwise = fromIntegral (h - 160) / 30.0

young :: Int -> Double
young a
  | a < 20    = 1.0
  | a > 50    = 0.0
  | otherwise = fromIntegral (50 - a) / 30.0

lukaImpl :: Double -> Double -> Double
lukaImpl a b = min 1.0 (1 - a + b)

defuzzCentroid :: [(Double, Double)] -> Double
defuzzCentroid pts =
  let n = sum [ x * m | (x, m) <- pts ]
      d = sum [ m     | (_, m) <- pts ]
  in  if d == 0 then 0 else n / d

demo_sec4 :: IO ()
demo_sec4 = do
  putStrLn "\n=== S4: Fuzzy Mathematics ==="
  putStrLn "\n-- tall(h):"
  mapM_ (\h -> putStrLn ("  tall(" ++ show h ++ "cm) = " ++ show (tall h)))
         [155, 165, 175, 185, 195 :: Int]
  putStrLn "\n-- T-norms (0.7, 0.6):"
  putStrLn ("  min="  ++ show (tMin  0.7 0.6))
  putStrLn ("  prod=" ++ show (tProd 0.7 0.6))
  putStrLn ("  luka=" ++ show (tLuka 0.7 0.6))
  let ht = 175 :: Int
      ag = 30  :: Int
  putStrLn "\n-- h=175 age=30:"
  putStrLn ("  AND(min)  = " ++ show (tMin  (tall ht) (young ag)))
  putStrLn ("  AND(prod) = " ++ show (tProd (tall ht) (young ag)))
  putStrLn ("  L-impl(0.8->0.6) = " ++ show (lukaImpl 0.8 0.6))
  let pts = [(fromIntegral x, tall x) | x <- [160, 165..190 :: Int]]
  putStrLn ("  centroid = " ++ show (defuzzCentroid pts) ++ " cm")

demo_sec4

# 5. Модальные Логики

Box=необходимость, Diamond=возможность. Kripke-семантика.

| S4 | `Box p->Box Box p` | транз. R |
|---|---|---|
| S5 | `Dia p->Box Dia p` | эквив. R |

### 🔸 Box = прав. сопряж. функтор

In [ ]:
-- S5: Modal Logics

data KripkeFrame w = KripkeFrame { worlds :: [w], access :: w -> [w] }

data Modal w
  = Atom    (w -> Bool)
  | MNot    (Modal w)
  | MAnd    (Modal w) (Modal w)
  | MOr     (Modal w) (Modal w)
  | MImpl   (Modal w) (Modal w)
  | Box     (Modal w)
  | Diamond (Modal w)

eval :: KripkeFrame w -> w -> Modal w -> Bool
eval _ w (Atom p)    = p w
eval f w (MNot x)    = not (eval f w x)
eval f w (MAnd x y)  = eval f w x && eval f w y
eval f w (MOr  x y)  = eval f w x || eval f w y
eval f w (MImpl x y) = not (eval f w x) || eval f w y
eval f w (Box x)     = all (\v -> eval f v x) (access f w)
eval f w (Diamond x) = any (\v -> eval f v x) (access f w)

checkAll :: Show w => KripkeFrame w -> Modal w -> [(w, Bool)]
checkAll f m = [ (w, eval f w m) | w <- worlds f ]

linearFrame :: KripkeFrame Int
linearFrame = KripkeFrame { worlds = [0..3], access = \w -> filter (>= w) [0..3] }

reflexiveFrame :: KripkeFrame Int
reflexiveFrame = KripkeFrame { worlds = [0..4], access = \w -> [w] ++ [w+1 | w < 4] }

demo_sec5 :: IO ()
demo_sec5 = do
  putStrLn "\n=== S5: Modal Logics ==="
  let ev = Atom even
      bx = Box ev
      di = Diamond ev
  putStrLn "\n-- Box(even) in S4 linear frame:"
  mapM_ (\(w,b) -> putStrLn ("  w=" ++ show w ++ " " ++ show b)) (checkAll linearFrame bx)
  putStrLn "\n-- Diamond(even):"
  mapM_ (\(w,b) -> putStrLn ("  w=" ++ show w ++ " " ++ show b)) (checkAll linearFrame di)
  putStrLn "\n-- T axiom Box(>1)->(>1) at w=2:"
  let p  = Atom (> 1)
      ax = MImpl (Box p) p
  putStrLn ("  result: " ++ show (eval reflexiveFrame 2 ax))
  putStrLn "\n-- Box(even) AND Dia(odd):"
  let f1 = MAnd (Box (Atom even)) (Diamond (Atom odd))
  mapM_ (\(w,b) -> putStrLn ("  w=" ++ show w ++ " => " ++ show b)) (checkAll linearFrame f1)

demo_sec5

# 6. Стохастическое Программирование: DSL

Стох. программа = морфизм в **Kleisli(Dist)**. Bayesian = disintegration.

### 🔸 Prior x Likelihood = Posterior (правило Байеса)

In [ ]:
-- S6: Stochastic DSL (Sprinkler Bayesian Network)

data Cloudy    = Cloudy    | NotCloudy    deriving (Show, Eq, Ord)
data Sprinkler = SprOn     | SprOff       deriving (Show, Eq, Ord)
data Rain      = Raining   | NotRaining   deriving (Show, Eq, Ord)
data WetGrass  = Wet       | Dry          deriving (Show, Eq, Ord)

priorCloudy :: Dist Cloudy
priorCloudy = Dist [(Cloudy, 0.5), (NotCloudy, 0.5)]

likelihoodSpr :: Cloudy -> Dist Sprinkler
likelihoodSpr Cloudy    = Dist [(SprOn, 0.1), (SprOff, 0.9)]
likelihoodSpr NotCloudy = Dist [(SprOn, 0.5), (SprOff, 0.5)]

likelihoodRain :: Cloudy -> Dist Rain
likelihoodRain Cloudy    = Dist [(Raining, 0.8), (NotRaining, 0.2)]
likelihoodRain NotCloudy = Dist [(Raining, 0.2), (NotRaining, 0.8)]

likelihoodWet :: Sprinkler -> Rain -> Dist WetGrass
likelihoodWet SprOn  Raining    = Dist [(Wet, 0.99), (Dry, 0.01)]
likelihoodWet SprOn  NotRaining = Dist [(Wet, 0.90), (Dry, 0.10)]
likelihoodWet SprOff Raining    = Dist [(Wet, 0.90), (Dry, 0.10)]
likelihoodWet SprOff NotRaining = Dist [(Wet, 0.00), (Dry, 1.00)]

jointDist :: Dist (Cloudy, Sprinkler, Rain, WetGrass)
jointDist = do
  c <- priorCloudy
  s <- likelihoodSpr c
  r <- likelihoodRain c
  w <- likelihoodWet s r
  return (c, s, r, w)

posteriorRain :: Dist Rain
posteriorRain = fmap (\(_,_,r,_) -> r) $ condition (\(_,_,_,w) -> w == Wet) jointDist

priorLinReg :: Dist (Double, Double)
priorLinReg = do
  a <- uniform [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
  b <- uniform [-2.0, -1.0, 0.0, 1.0, 2.0]
  return (a, b)

demo_sec6 :: IO ()
demo_sec6 = do
  putStrLn "\n=== S6: Stochastic Programming ==="
  putStrLn "\n-- P(Rain | WetGrass=Wet):"
  putStr (showDist posteriorRain)
  putStrLn "\n-- P(WetGrass):"
  putStr (showDist (fmap (\(_,_,_,w) -> w) jointDist))
  putStrLn "\n-- Prior regression (slope, intercept) x3:"
  mapM_ (\((a,b),p) -> putStrLn
    ("  (" ++ show a ++ ", " ++ show b ++ ") p="
     ++ show (round (p*100) :: Int) ++ "%"))
    (take 3 (getProbs priorLinReg))

demo_sec6

# 7. Марковские Цепи в Категории Клейсли

Markov chain = морфизм `S -> Dist S` в Kleisli(Dist).

### 🔸 Стационарное распред. = **initial algebra** для F(X)=Dist(X)

In [ ]:
-- S7: Markov Chains via Kleisli

type TransKernel s = s -> Dist s

runChain :: Int -> TransKernel s -> Dist s -> Dist s
runChain 0 _ d = d
runChain n k d = runChain (n-1) k (d >>= k)

normalizeDist :: Ord a => Dist a -> Dist a
normalizeDist (Dist ps) = Dist (Map.toList (Map.fromListWith (+) ps))

data Weather7 = Sunny7 | Cloudy7 | Rainy7 deriving (Show, Eq, Ord)

wk :: TransKernel Weather7
wk Sunny7  = Dist [(Sunny7, 0.7), (Cloudy7, 0.2), (Rainy7, 0.1)]
wk Cloudy7 = Dist [(Sunny7, 0.3), (Cloudy7, 0.4), (Rainy7, 0.3)]
wk Rainy7  = Dist [(Sunny7, 0.2), (Cloudy7, 0.3), (Rainy7, 0.5)]

gamblerK :: Int -> Dist Int
gamblerK 0 = Dist [(0, 1.0)]
gamblerK 4 = Dist [(4, 1.0)]
gamblerK n = Dist [(n-1, 0.4), (n+1, 0.6)]

demo_sec7 :: IO ()
demo_sec7 = do
  putStrLn "\n=== S7: Markov Chains ==="
  let d0 = certainly Sunny7
  mapM_ (\n -> do
    let nd = normalizeDist (runChain n wk d0)
    putStrLn ("  step " ++ show n ++ ":")
    mapM_ (\(w,p) -> putStrLn
      ("    " ++ show w ++ " = " ++ show (round (p*100) :: Int) ++ "%"))
      (getProbs nd)) [1,2,5]
  let uni  = uniform [Sunny7, Cloudy7, Rainy7]
      stat = last $ take 20 $ iterate (normalizeDist . (>>= wk)) uni
  putStrLn "\n-- Stationary distribution:"
  mapM_ (\(w,p) -> putStrLn
    ("  " ++ show w ++ " = " ++ show (round (p*1000) :: Int) ++ "/1k"))
    (getProbs stat)
  let gn = normalizeDist (runChain 5 gamblerK (certainly 2))
  putStrLn "\n-- Gambler from 2 (5 steps):"
  mapM_ (\(s,p) -> putStrLn
    ("  s=" ++ show s ++ " p=" ++ show (round (p*100) :: Int) ++ "%"))
    (getProbs gn)

demo_sec7

# 8. Стохастические Матрицы и Теория Меры

| Кат. | Объекты | Морф. |
|-----|----|-----|
| Hask | типы | чистые функ. |
| Kleisli(Dist) | типы | `a->Dist b` |
| FinStoch | кон. мн-ва | стох. матр |

**Монада** = (Dist, return, join). **Энтропия** — ест. инвариант.

In [ ]:
-- S8: Categorical Picture

joinDist :: Dist (Dist a) -> Dist a
joinDist (Dist outer) = Dist
  [ (x, p * q)
  | (inner, p) <- outer
  , (x, q)     <- getProbs inner
  ]

entropy :: Dist a -> Double
entropy (Dist ps) = negate (sum [ p * logBase 2 p | (_, p) <- ps, p > 0 ])

klDiv :: Ord a => Dist a -> Dist a -> Double
klDiv (Dist ps) (Dist qs) =
  sum [ p * logBase 2 (p / Map.findWithDefault 1e-10 a (Map.fromList qs))
      | (a, p) <- ps, p > 0 ]

type StochMatrix s = Map (s,s) Double

mkStoch :: Ord s => [(s, s, Double)] -> StochMatrix s
mkStoch ts = Map.fromList [ ((a,b), p) | (a,b,p) <- ts ]

applyStoch :: Ord s => StochMatrix s -> [s] -> Dist s -> Dist s
applyStoch mat ss d =
  d >>= (\s -> Dist [ (t, Map.findWithDefault 0 (s,t) mat) | t <- ss ])

demo_sec8 :: IO ()
demo_sec8 = do
  putStrLn "\n=== S8: Categorical Picture ==="
  putStrLn ("  H(die)     = " ++ show (entropy (uniform [1..6::Int])))
  putStrLn ("  H(certain) = " ++ show (entropy (certainly True)))
  putStrLn ("  H(coin)    = " ++ show (entropy (uniform [True, False])))
  let p1 = Dist [(1::Int, 0.5), (2, 0.5)]
      p2 = Dist [(1::Int, 0.8), (2, 0.2)]
  putStrLn ("  KL(fair|biased) = " ++ show (klDiv p1 p2))
  putStrLn ("  KL(biased|fair) = " ++ show (klDiv p2 p1))
  let d1 = Dist [(uniform [1,2::Int], 0.3), (uniform [3,4::Int], 0.7)]
  putStrLn ("  join: " ++ show (joinDist d1))
  let mat = mkStoch [(1,1,0.9),(1,2,0.1),(2,1,0.3),(2,2,0.7::Double)]
      nd  = normalizeDist
              (applyStoch mat [1,2] (applyStoch mat [1,2] (certainly (1::Int))))
  putStrLn "  FinStoch 2-state:"
  putStr (showDist nd)

demo_sec8

# 9. Теория Субъективного Моделирования Пытьева

> Эта тема вынесена в отдельный ноутбук: **[SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)**

## Краткое введение

**Субъективная модель** случайной величины — альтернатива вероятностной модели для ситуаций неполного знания. Вместо одной вероятностной меры вводятся **две дуальные меры**:

- **Правдоподобность** Pl(E) = max степень доверия к событию E
- **Достоверность** Bel(E) = min степень уверенности в событии E
- **Зазор** Pl(E) - Bel(E) — мера незнания

Теория разработана Ю.П. Пытьевым (МГУ). Подробное изложение с полными формулами, реализацией на Haskell и примером диагностики двигателя с тремя экспертами — в **[SubjectiveModeling.ipynb](SubjectiveModeling.ipynb)**.

| Понятие | Формула |
|---------|----------|
| Правдоподобность события E | Pl(E) = max по x из E значений t(x) |
| Достоверность события E | Bel(E) = min по x вне E значений tau(x) |
| Интеграл Пытьева (оптимист) | sup min{ t(x), f(x) } |
| Интеграл Пытьева (пессимист) | inf max{ tau(x), g(x) } |


In [ ]:
-- S9: Теория Субъективного Моделирования Пытьева
-- Источник: Вестник МГУ, Сер.1, 2018. Математические и эмпирические основы.

-- ============================================================
-- 9.1 Базовые типы субъективной модели
-- ============================================================

-- Дистрибутив: список (значение, мера t(x)) или (значение, мера tau(x))
type Dist a = [(a, Double)]

-- Субъективная модель (X, t, tau): t(x) = Pl(x~=x), tau(x) = Bel(x~≠x)
data SubjModel a = SubjModel
  { domain   :: [a]          -- X — множество значений
  , plDist   :: Dist a       -- t(x): pl-распределение
  , belDist  :: Dist a       -- tau(x): bel-распределение
  } deriving (Show)

-- ============================================================
-- 9.2 Pl и Bel меры — точные формулы из первоисточника
-- Pl(E) = sup_{x in E} t(x)
-- Bel(E) = inf_{x not in E} tau(x)
-- ============================================================

plMeasure :: Eq a => SubjModel a -> [a] -> Double
plMeasure m event
  | null event = 0.0
  | otherwise  = maximum [v | (x, v) <- plDist m, x `elem` event]

belMeasure :: Eq a => SubjModel a -> [a] -> Double
belMeasure m event =
  let complement = filter (`notElem` event) (domain m)
  in if null complement
     then 1.0
     else minimum [v | (x, v) <- belDist m, x `elem` complement]

-- ============================================================
-- 9.3 Pl- и Bel-интегралы (обобщённые интегралы Сугено)
-- pl_t(f)   = sup_{x in X} min{ t(x), f(x) }
-- bel_tau(g) = inf_{x in X} max{ tau(x), g(x) }
-- ============================================================

plIntegral :: Dist a -> (a -> Double) -> Double
plIntegral dist f = maximum [min t (f x) | (x, t) <- dist]

belIntegral :: Dist a -> (a -> Double) -> Double
belIntegral dist g = minimum [max tau (g x) | (x, tau) <- dist]

-- ============================================================
-- 9.4 Предельные модели
-- ============================================================

-- Абсолютное незнание: t(x)=1 для всех x, tau(x)=0 для всех x
absoluteIgnorance :: [a] -> SubjModel a
absoluteIgnorance xs = SubjModel
  { domain  = xs
  , plDist  = [(x, 1.0) | x <- xs]
  , belDist = [(x, 0.0) | x <- xs]
  }

-- Точное знание x0: t(x0)=1, t(x)=0; tau(x0)=1, tau(x)=0
exactKnowledge :: Eq a => a -> [a] -> SubjModel a
exactKnowledge x0 xs = SubjModel
  { domain  = xs
  , plDist  = [(x, if x == x0 then 1.0 else 0.0) | x <- xs]
  , belDist = [(x, if x == x0 then 1.0 else 0.0) | x <- xs]
  }

-- ============================================================
-- 9.5 Принцип относительности — группа Gamma
-- Естественное преобразование: применить gamma in Gamma к модели
-- ============================================================

-- Тип автоморфизма шкалы: gamma: [0,1] -> [0,1], монотонный
type ScaleAuto = Double -> Double

-- Применить автоморфизм к pl-распределению
applyGamma :: ScaleAuto -> SubjModel a -> SubjModel a
applyGamma g m = m
  { plDist  = [(x, g v) | (x, v) <- plDist m]
  , belDist = [(x, g v) | (x, v) <- belDist m]
  }

-- Нормировать pl-распределение чтобы sup t(x) = 1
normalizePl :: SubjModel a -> SubjModel a
normalizePl m =
  let maxT = maximum (map snd (plDist m))
  in if maxT == 0
     then m
     else m { plDist = [(x, v / maxT) | (x, v) <- plDist m] }

-- ============================================================
-- 9.6 Условие двойственной согласованности
-- Exists theta: Bel(A) = theta(Pl(X\A))  forall A
-- Проверяем приближённо: корреляция Bel(A) и 1-Pl(X\A)
-- ============================================================

isDuallyConsistent :: Eq a => SubjModel a -> Bool
isDuallyConsistent m =
  let xs  = domain m
      -- генерируем все подмножества (для малых X)
      subsets [] = [[]]
      subsets (y:ys) = let ss = subsets ys in ss ++ map (y:) ss
      allSets = subsets xs
      pairs = [(belMeasure m a, plMeasure m (filter (`notElem` a) xs))
               | a <- allSets]
      -- Проверяем монотонность: если pl(X\A) растёт, bel(A) убывает
      sorted = map fst (filter (\(b,p) -> p > 0 && p < 1) pairs)
      -- Упрощённая проверка: bel <= 1 - pl (всегда выполняется)
      check (b, p) = b <= 1.0 - p + 1e-10
  in all check pairs

-- ============================================================
-- 9.7 Комбинирование экспертных оценок
-- Взвешенная сумма с весами w_i >= 0, sum w_i = 1
-- ============================================================

-- Вход: список (модель, вес)
combineExperts :: Eq a => [(SubjModel a, Double)] -> SubjModel a
combineExperts experts =
  let totalW = sum (map snd experts)
      ws     = map (\(m, w) -> (m, w / totalW)) experts
      xs     = domain (fst (head experts))
      combineDist getter =
        [(x, sum [w * maybe 0 id (lookup x (getter m))
                 | (m, w) <- ws])
        | x <- xs]
      combined = SubjModel
        { domain  = xs
        , plDist  = combineDist plDist
        , belDist = combineDist belDist
        }
  in normalizePl combined

-- ============================================================
-- 9.8 Условная субъективная модель
-- Условие: пересечь domain с наблюдаемым событием
-- ============================================================

conditionModel :: Eq a => SubjModel a -> [a] -> SubjModel a
conditionModel m observed =
  let newDom = filter (`elem` observed) (domain m)
      restrict d = [(x, v) | (x, v) <- d, x `elem` newDom]
      conditioned = SubjModel
        { domain  = newDom
        , plDist  = restrict (plDist m)
        , belDist = restrict (belDist m)
        }
  in normalizePl conditioned

-- ============================================================
-- 9.9 Pl-независимость двух компонент
-- Pl(x1, x2) = min{t1(x1), t2(x2)}
-- ============================================================

jointPlDist :: Dist a -> Dist b -> Dist (a, b)
jointPlDist d1 d2 =
  [((x1, x2), min t1 t2) | (x1, t1) <- d1, (x2, t2) <- d2]

jointBelDist :: Dist a -> Dist b -> Dist (a, b)
jointBelDist d1 d2 =
  [((x1, x2), max tau1 tau2) | (x1, tau1) <- d1, (x2, tau2) <- d2]

-- ============================================================
-- 9.10 ДЕМО: Диагностика двигателя — 3 эксперта
-- ============================================================

-- Состояния двигателя
data EngineState = OK | WarnFilter | WarnBearing | Critical
  deriving (Show, Eq, Ord)

allStates :: [EngineState]
allStates = [OK, WarnFilter, WarnBearing, Critical]

-- Три эксперта дают оценки t(x) (правдоподобность каждого состояния)
expert1 :: SubjModel EngineState
expert1 = normalizePl $ SubjModel
  { domain  = allStates
  , plDist  = [(OK, 0.3), (WarnFilter, 0.9), (WarnBearing, 0.5), (Critical, 0.1)]
  , belDist = [(OK, 0.1), (WarnFilter, 0.0), (WarnBearing, 0.2), (Critical, 0.6)]
  }

expert2 :: SubjModel EngineState
expert2 = normalizePl $ SubjModel
  { domain  = allStates
  , plDist  = [(OK, 0.2), (WarnFilter, 0.7), (WarnBearing, 0.8), (Critical, 0.3)]
  , belDist = [(OK, 0.2), (WarnFilter, 0.1), (WarnBearing, 0.0), (Critical, 0.5)]
  }

expert3 :: SubjModel EngineState
expert3 = normalizePl $ SubjModel
  { domain  = allStates
  , plDist  = [(OK, 0.1), (WarnFilter, 0.6), (WarnBearing, 0.9), (Critical, 0.4)]
  , belDist = [(OK, 0.3), (WarnFilter, 0.2), (WarnBearing, 0.0), (Critical, 0.4)]
  }

-- Комбинированная модель
combinedModel :: SubjModel EngineState
combinedModel = combineExperts
  [(expert1, 0.4), (expert2, 0.35), (expert3, 0.25)]

-- Условная модель: исключаем Critical (датчик сказал "не критично")
conditioned :: SubjModel EngineState
conditioned = conditionModel combinedModel [OK, WarnFilter, WarnBearing]

-- Функция потерь для интеграла Сугено: серьёзность состояния
severity :: EngineState -> Double
severity OK           = 0.0
severity WarnFilter   = 0.3
severity WarnBearing  = 0.6
severity Critical     = 1.0

demo_sec9 :: IO ()
demo_sec9 = do
  putStrLn "=== Теория Субъективного Моделирования Пытьева ==="
  putStrLn ""

  -- 1. Предельные модели
  let ign  = absoluteIgnorance allStates
      know = exactKnowledge WarnBearing allStates
  putStrLn "--- Предельные модели ---"
  putStrLn $ "Абсолютное незнание: Pl(любое)=" ++
             show (plMeasure ign [OK, WarnFilter])
  putStrLn $ "Абсолютное незнание: Bel(пусто)=" ++
             show (belMeasure ign [])
  putStrLn $ "Точное знание WarnBearing: Pl({WB})=" ++
             show (plMeasure know [WarnBearing])
  putStrLn $ "Точное знание WarnBearing: Pl({OK})=" ++
             show (plMeasure know [OK])
  putStrLn ""

  -- 2. Принцип относительности (gamma = sqrt)
  let sqrtModel = applyGamma sqrt expert1
  putStrLn "--- Принцип относительности (gamma = sqrt) ---"
  putStrLn $ "До гамма: Pl({WarnFilter})=" ++
             show (plMeasure expert1 [WarnFilter])
  putStrLn $ "После gamma=sqrt: Pl({WarnFilter})=" ++
             show (plMeasure sqrtModel [WarnFilter])
  putStrLn "  (порядок сохранён — принцип относительности)"
  putStrLn ""

  -- 3. Интегралы Пытьева-Сугено
  putStrLn "--- Интегралы Пытьева (обобщ. Сугено) ---"
  let plRisk  = plIntegral  (plDist combinedModel)  severity
      belRisk = belIntegral (belDist combinedModel) severity
  putStrLn $ "Pl-интеграл риска (оптимист): " ++ show plRisk
  putStrLn $ "Bel-интеграл риска (пессимист): " ++ show belRisk
  putStrLn $ "Разрыв (зазор неопределённости): " ++ show (plRisk - belRisk)
  putStrLn ""

  -- 4. Меры событий
  let warn = [WarnFilter, WarnBearing]
  putStrLn "--- Комбинированная модель 3 экспертов ---"
  putStrLn $ "Pl({WarnFilter, WarnBearing}) = " ++
             show (plMeasure combinedModel warn)
  putStrLn $ "Bel({WarnFilter, WarnBearing}) = " ++
             show (belMeasure combinedModel warn)
  putStrLn $ "Зазор Pl-Bel: " ++
    show (plMeasure combinedModel warn - belMeasure combinedModel warn)
  putStrLn ""

  -- 5. Условная модель
  putStrLn "--- Условная модель (без Critical) ---"
  putStrLn $ "Pl(WarnBearing | ¬Critical) = " ++
             show (plMeasure conditioned [WarnBearing])
  putStrLn $ "Pl(WarnFilter | ¬Critical) = " ++
             show (plMeasure conditioned [WarnFilter])
  putStrLn ""

  -- 6. Двойственная согласованность
  putStrLn "--- Двойственная согласованность ---"
  putStrLn $ "Эксперт1: Bel <= 1-Pl? " ++ show (isDuallyConsistent expert1)
  putStrLn $ "Объединённая: Bel <= 1-Pl? " ++ show (isDuallyConsistent combinedModel)

demo_sec9

## Сводка

Рассмотрены 9 структур: неопределённость как функтор, монада дискретных распределений, монада Гири и марковские ядра, нечёткая математика и нечёткие решётки, модальные логики и фреймы Крипке, стохастическое программирование с байесовскими сетями, марковские цепи в категории Клейсли, категорная картина стохастических матриц и теории меры, и **теория субъективного моделирования Пытьева** (вынесена в отдельный ноутбук).

| № | Тема | Ключевая идея |
|---|------|---------------|
| 1 | Функторы неопределённости | Maybe, Either, список, Validation |
| 2 | Монада дискретных распределений | Байесовское обновление, дискретные вероятности |
| 3 | Монада Гири | Марковские ядра, вероятностные меры на пространстве |
| 4 | Нечёткая математика | Треугольные нормы, нечёткие множества, решётки |
| 5 | Модальные логики | Операторы Box и Diamond, фреймы Крипке |
| 6 | Стохастическое программирование | Байесовские сети, предметно-ориентированный язык |
| 7 | Марковские цепи | Цепи в категории Клейсли, стационарные распределения |
| 8 | Стохастические матрицы и теория меры | Энтропия, дивергенция Кульбака-Лейблера |
| 9 | Теория Пытьева | Правдоподобность, достоверность, субъективные модели |


---

**← Предыдущий:** [Comonads](Comonads.ipynb) | **→ Следующий:** [AlgebrasCoalgebras](AlgebrasCoalgebras.ipynb)